In [1]:
import os
import sys

import logging

%pwd
os.chdir("../")

# Add the absolute path to src/ so Python can find automatch
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.append(src_path)
    
%pwd


from src.onBench.utils.EDA import *


logging.getLogger("snowflake").setLevel(logging.WARNING)
logging.getLogger("snowflake.connector").setLevel(logging.WARNING)
logging.getLogger("snowflake.snowpark").setLevel(logging.WARNING)


from src.onBench.entity.config_entity import DataIngestionConfig
from src.onBench.config.configuration import ConfigurationManager



c:\Users\fiscarelli\conda_venvs\onbench_venv\lib\site-packages\holoviews\__init__.py:81: FutureWarning: param.version.Version has been deprecated and will be removed in a future version.
  __version__ = str(param.version.Version(fpath=__file__, archive_commit="$Format:%h$",
c:\Users\fiscarelli\conda_venvs\onbench_venv\lib\site-packages\param\version.py:200: FutureWarning: param.version.run_cmd has been deprecated and will be removed in a future version.
  remotes = run_cmd([cmd, 'remote', '-v'],
c:\Users\fiscarelli\conda_venvs\onbench_venv\lib\site-packages\panel\config.py:35: FutureWarning: param.version.Version has been deprecated and will be removed in a future version.
  __version__ = str(param.version.Version(
c:\Users\fiscarelli\conda_venvs\onbench_venv\lib\site-packages\param\version.py:200: FutureWarning: param.version.run_cmd has been deprecated and will be removed in a future version.
  remotes = run_cmd([cmd, 'remote', '-v'],
c:\Users\fiscarelli\conda_venvs\onbench_venv\lib\

In [2]:
from src.onBench.utils.snowflake_utils import get_snowpark_session
session = get_snowpark_session()

In [3]:
df = session.sql(f"""select *
                     from it_discovery.ita_exp_onbench.MPG_IT_ONBENCH_TEST_CANDIDATE_CLEANED--_ALL_EDA
                     --where status = 'No mission'
                 """).to_pandas()



In [4]:
df["CCNL_CODE"] = df["CCNL_CODE"].astype("Int64").astype("string")

df[df.select_dtypes(include=["object", "string"]).columns] = (
    df.select_dtypes(include=["object", "string"]).apply(lambda col: col.str.strip())
)


In [5]:
keywords = [

    #non utilizzare
    #"curriculum", "cv", "vitae", 

    #generali
    "esperienza", "lavoro", "occupazione", "posizione", "ruolo", "azienda",
    "experience", "job", "position"

    # ruoli
    "addetto", "operaio", "impiegato", "responsabile", "cassiere", "magazziniere",
    "analista", "sviluppatore", "developer", "autista", "segretaria", "tecnico", "consulente",
    "worker", "manager", "coordinator", "assistant", "engineer", "specialist", "warehouse", "driver", "operator", "supervisor", "administrator", "consultant"

    # titoli di studio
    "diploma", "laurea", "master", "istituto", "universit",
    "degree", "master", "university", "school", "college",

    # competenze
    "excel", "office", "inglese", "informatica", "patente",
    "english", "computer", "driving license"

    # pattern temporali
    #"2019", "2020", "2021", "2022", "2023", "2024", "dal", "al",

    # aziende / enti
    "srl", "spa", "cooperativa", "comune", "regione"
    "regione", "municipality", "company", "organization", "corporation"
]

'''
df["IS_VALID_CV"] = df["CURRICULUM"].str.lower().apply(
    lambda x: any(k in x for k in keywords) and len(x) > 50 if isinstance(x, str) else False
)

is_valid_cv = np.sum(df["IS_VALID_CV"])/df.shape[0]

print(f"""Percentage of invalid CVs (custom keywords matching): {1-is_valid_cv:.2%}""")
'''



'\ndf["IS_VALID_CV"] = df["CURRICULUM"].str.lower().apply(\n    lambda x: any(k in x for k in keywords) and len(x) > 50 if isinstance(x, str) else False\n)\n\nis_valid_cv = np.sum(df["IS_VALID_CV"])/df.shape[0]\n\nprint(f"""Percentage of invalid CVs (custom keywords matching): {1-is_valid_cv:.2%}""")\n'

# MISSING CVs #

Other than using CV length to identify real CVs (50 characters), I also define a set of keywords that are normally found in a CV and I check whether a CV contains any. ù
So, if the CV contains at least 50 characters and at least one of the keywords mentioned, I classify it as real.
According to this method, 47% of candidate CS are not valid (are emtpy or contain meaningless text)

In [6]:
general_info(df)


Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 457 entries, 0 to 456
Data columns (total 33 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   CANDIDATE_ID_BH        457 non-null    int32  
 1   DAYS_ONBENCH_LAST      158 non-null    float64
 2   DAYS_ONBENCH_TOT       457 non-null    int16  
 3   DAYS_ONBENCH_PERC      457 non-null    int8   
 4   PARTTIME_PERC          457 non-null    float64
 5   CANDIDATE_ID_PB        457 non-null    object 
 6   FIRST_NAME             457 non-null    object 
 7   LAST_NAME              457 non-null    object 
 8   CAP                    441 non-null    object 
 9   DESCRIPTION            350 non-null    object 
 10  JOB_SITE               457 non-null    object 
 11  JOB_FIRST              457 non-null    object 
 12  JOB_INTERMEDIATE       182 non-null    object 
 13  JOB_LAST               308 non-null    object 
 14  CCNL_LEVEL             457 non-null    object 
 15 

# COMMENTS on missing/unique values #

9085 candidates

Comments on candidate missing data (Bullhorn):
-   salary (99% missing)
-   turno_preferenza (99.5% missing)
-   parttime_preferenza_perc (99.7% missing)
-   skills (82% missing)
-   education (66% missing)
-   certifications (100% missing)
-   curriculum 48% missing
-   description (34% missing. this means that curriculum, skills, education and certifiations are all simultaneaously missing)

Comments on candidate missing data (PowerBase):
-   parttime_perc (first job) (88% missing)
-   CCNL level (first job) (0% missing)
-   CCNL code (first job) (1.4%)
-   CCNL description (first job) (1.4%)
-   legal_category (first job) (0% missing)


Some comments on unique values:
- 11 unique values for education
- 776 unique first jobs
- 172 unique CCNL codes
- 209 unique CCNL descriptions
- 6 unique legal categories

In [8]:
"""
columns_to_check = ["EDUCATION", "CERTIFICATIONS"]

# Print unique values for each column
for col in columns_to_check:
    print(f"Unique values in '{col}':")
    print(df[col].sort_values().unique())
    print("-" * 40)
"""

'\ncolumns_to_check = ["EDUCATION", "CERTIFICATIONS"]\n\n# Print unique values for each column\nfor col in columns_to_check:\n    print(f"Unique values in \'{col}\':")\n    print(df[col].sort_values().unique())\n    print("-" * 40)\n'

In [9]:
columns_to_check = ["CCNL_LEVEL", "CCNL_CODE", "LEGAL_CATEGORY", "CCNL_DESCRIPTION"]

# Print unique values for each column
for col in columns_to_check:
    print(f"Unique values in '{col}':")
    print(df[col].sort_values().unique())
    print("-" * 40)



Unique values in 'CCNL_LEVEL':
['1' '2' '2tr' '3' '4' '4 junior' '4 senior' '4 super (ex 5)' '5'
 '5 (ex 6)' '6' '6 junior' '6 senior' '6 super (ex 7)' '7' '8' 'a' 'ae1'
 'ap prima' 'area 3 liv. 1' 'area3liv.1' 'b' 'b/1' 'b1' 'b2' 'c/1' 'c/2'
 'c/3' 'c1' 'c2' 'c3' 'comune' 'd' 'd/1' 'd/2' 'd1' 'd1 (ex 6)'
 'd1 (ex d2)' 'd2' 'd3' 'e' 'e1' 'e2' 'e3' 'e4' 'f' 'g' 'g1' 'h' 'i'
 'op base' 'op centrato']
----------------------------------------
Unique values in 'CCNL_CODE':
<StringArray>
['1020',  '106', '1250', '1270', '1380', '1390', '1620',   '17', '1790',
 '1810', '1820', '1870',   '20', '2070',  '220', '2220',  '230', '2360',
  '240', '2470', '2630', '2930',  '330',   '38',   '39', '3980',  '700',
  '710',  '720',   '81',   '88',   '89', '9013', '9050', '9332', '9335',
 '9354', '9357', '9408', '9504', '9632', '9633', '9646', '9659', '9662',
 '9674', '9683', '9719', '9726', '9746', '9757', '9776', '9789', '9791',
 '9796', '9808', '9816',   <NA>]
Length: 58, dtype: string
----------------

In [10]:
columns_to_check = ["JOB_FIRST"]

# Print unique values for each column
for col in columns_to_check:
    print(f"Unique values in '{col}':")
    print(df[col].sort_values().unique())
    print("-" * 40)

Unique values in 'JOB_FIRST':
['add. rifinitura- qualità e confez. pelletteria' 'addetti global custody'
 'addetto agli allestimenti interni ferroviari' 'addetto agli scaffali'
 'addetto ai controlli contabili' 'addetto al magazzino'
 'addetto al magazzino settore abbigliamento' 'addetto al montaggio'
 'addetto al ricevimento delle materie prime'
 'addetto alimentazione mulini' 'addetto alla fatturazione'
 'addetto alla formatura' 'addetto alla logistica'
 'addetto alla macchina di laminazione della carta'
 'addetto alla pescheria' 'addetto alla pianificazione acquisti'
 'addetto alla pressa per lo stampaggio'
 'addetto alla produzione di gomme da masticare'
 'addetto alla produzione di pasta alimentare' 'addetto alla reception'
 'addetto alla rifinitura di prodotti di pelletteria'
 'addetto alla rifinizione'
 'addetto alla vendita attrezzature audio video'
 'addetto alla vendita dei biglietti' 'addetto alle pulizie'
 'addetto alle pulizie negli stabili'
 'addetto all’assemblaggio di m

In [11]:
columns_to_check = ["JOB_SITE"]

# Print unique values for each column
for col in columns_to_check:
    print(f"Unique values in '{col}':")
    print(df[col].sort_values().unique())
    print("-" * 40)

Unique values in 'JOB_SITE':
['c.da piano  saletti snc paglieta (ch)'
 'c/o cantieri tratto autostradale savona - vantimiglia savona (sv)'
 'c/o chemi spa via vadisi patrica (fr)'
 'c/o dyson demo store - coin napoli napoli (na)'
 'c/o fiat sata- z.i. san nicola di melfi (pz) melfi (pz)'
 'c/o hitachi rail pistoia pistoia (pt)'
 'c/o mdt missiondreamtravel    via germanico 45    roma (rm)'
 'c/o ospedale m. bufalini cesena (fc)'
 'c/o pastificio de cecco via f. de cecco fara san martino (ch)'
 'c/o preziosi food s.r.l. zona ind. san nicola di melfi melfi melfi (pz)'
 'c/o unicredit via iv novembre, 44 motta di livenza (tv)'
 'centro servizi océ c/o wartsila -bagnoli della rosandra, 334 san dorligo della valle (ts)'
 'cinema multisala ciaky, traversa via bitonto bari (ba)'
 'comune di bergamo - piazza vecchia, 8 bergamo (bg)'
 'contrada san luca  67 atessa (ch)'
 "contrada santa maria d'attoli ginosa (ta)"
 'coop estense- via giardini 77/79 formigine (mo)'
 'corso giuseppe garibaldi, 18

In [14]:
"""
b1 = bar_plot(df, 'EDUCATION', rotation=25)
b2 = bar_plot(df, 'CERTIFICATIONS', rotation=25)
b3 = bar_plot(df, 'IS_VALID_CV', rotation=25)
b4 = bar_plot(df, 'CANDIDATE_STATUS', rotation=25)


(b1 + b2 + b3 + b4).cols(1)
"""

"\nb1 = bar_plot(df, 'EDUCATION', rotation=25)\nb2 = bar_plot(df, 'CERTIFICATIONS', rotation=25)\nb3 = bar_plot(df, 'IS_VALID_CV', rotation=25)\nb4 = bar_plot(df, 'CANDIDATE_STATUS', rotation=25)\n\n\n(b1 + b2 + b3 + b4).cols(1)\n"

In [15]:
b1 = bar_plot(df, 'CAP', rotation=25)
b2 = bar_plot(df, 'CITY', rotation=25)
b3 = bar_plot(df, 'PROVINCE', rotation=25)

(b1 + b2 + b3).cols(1)

:Layout
   .Bars.I   :Bars   [CAP]   (COUNT)
   .Bars.II  :Bars   [CITY]   (COUNT)
   .Bars.III :Bars   [PROVINCE]   (COUNT)

In [16]:
b1 = bar_plot(df, 'CAP', rotation=25)
b2 = bar_plot(df, 'CITY', rotation=25)
b3 = bar_plot(df, 'JOB_SITE_PROVINCE', rotation=25)

display(b3)

:Bars   [JOB_SITE_PROVINCE]   (COUNT)

In [17]:
df[['JOB_SITE', 'JOB_SITE_NORM']]

,JOB_SITE,JOB_SITE_NORM
0,"via caduti e dispersi in russia, 2 sedico (bl)",via caduti e dispersi in russia 2 sedico bl
1,"viale bramante, 29 c/o centro commerciale parc...",viale bramante 29 c o centro commerciale parco...
2,"polo nave de vero ss romea, snc venezia (ve)",polo nave de vero ss romea snc venezia ve
3,ss 206 km 28 - loc. torretta vecchia fauglia (pi),ss 206 km 28 loc torretta vecchia fauglia pi
4,"via statale 12 nord, 86 mirandola (mo)",via statale 12 nord 86 mirandola mo
...,...,...
452,"via fratte,6968 sant'elpidio a mare (fm)",via fratte 6968 sant elpidio a mare fm
453,"via fratte,6968 sant'elpidio a mare (fm)",via fratte 6968 sant elpidio a mare fm
454,"viale dell'industria s.s. 7 km 0,500 - zona in...",viale dell industria s s 7 km 0 500 zona ind l...
455,via ganci e soletto snc zi termini imerese (pa),via ganci e soletto snc zi termini imerese pa


In [18]:
b1 = bar_plot(df, 'CCNL_LEVEL')#, label_range=30)
b2 = bar_plot(df, 'CCNL_CODE')#, label_range=30)
b3 = bar_plot(df, 'LEGAL_CATEGORY', rotation=25)#, label_range=30)
b4 = bar_plot(df, 'CCNL_DESCRIPTION', rotation=25, height=600, width=1200)

(b1 + b2 + b3 + b4).cols(1)

:Layout
   .Bars.I   :Bars   [CCNL_LEVEL]   (COUNT)
   .Bars.II  :Bars   [CCNL_CODE]   (COUNT)
   .Bars.III :Bars   [LEGAL_CATEGORY]   (COUNT)
   .Bars.IV  :Bars   [CCNL_DESCRIPTION]   (COUNT)

In [19]:
b1 = bar_plot(df, 'JOB_FIRST', rotation=25, height=600, width=1200)#, label_range=30)
b2 = bar_plot(df, 'JOB_LAST', rotation=25, height=600, width=1200)

(b1 + b2).cols(1)

:Layout
   .Bars.I  :Bars   [JOB_FIRST]   (COUNT)
   .Bars.II :Bars   [JOB_LAST]   (COUNT)

In [20]:
b1 = bar_plot(df, 'JOB_SITE', rotation=25, height=800, width=2400)#, label_range=30)
b1

:Bars   [JOB_SITE]   (COUNT)

In [21]:
h1 = hist_plot(df, "PARTTIME_PERC", bins=20, xrange=[], yrange=[], height=400, width=800)

h1

:Histogram   [PARTTIME_PERC]   (Frequency)

In [22]:
import scipy
print(scipy.__version__)

1.11.4


In [23]:
h1 = hist_plot(df, "DAYS_ONBENCH_LAST", bins=100, xrange=[], yrange=[], height=400, width=800)
h2 = hist_plot(df, "DAYS_ONBENCH_TOT", bins=100, xrange=[], yrange=[], height=400, width=800)
h3 = hist_plot(df, "DAYS_ONBENCH_PERC", bins=100, xrange=[], yrange=[], height=400, width=800)

(h1 + h2 + h3).cols(1)

:Layout
   .Histogram.I   :Histogram   [DAYS_ONBENCH_LAST]   (Frequency)
   .Histogram.II  :Histogram   [DAYS_ONBENCH_TOT]   (Frequency)
   .Histogram.III :Histogram   [DAYS_ONBENCH_PERC]   (Frequency)

# VACANCY #

In [24]:
df_vacancy = session.sql(f"""select *
                     from it_discovery.ita_exp_onbench.MPG_IT_ONBENCH_TEST_JOBORDER_CLEANED
                 """).to_pandas()



In [25]:
df_vacancy["RUOLO_LAVORATIVO_STANDARD_MPG_SPLIT"] = df_vacancy["RUOLO_LAVORATIVO_STANDARD_MPG"].str.split("/").str[0]


In [26]:
keywords = [

    #non utilizzare
    #"curriculum", "cv", "vitae", 

    #generali
    "ricerc", "cerc", "esperienza", "lavor", "occupazion", "posizion", "ruol", "aziend",
    "experience", "job", "position"

    # ruoli
    "addett", "operai", "impiegat", "responsabil", "cassier", "magazzinier",
    "analist", "sviluppator", "developer", "autist", "segretar", "tecnic", "consulent",
    "worker", "manager", "coordinator", "assistant", "engineer", "specialist", "warehouse", "driver", "operator", "supervisor", "administrator", "consultant"

    # titoli di studio
    "diplom", "laure", "master", "istitut", "universit",
    "degree", "master", "university", "school", "college",

    # competenze
    "excel", "office", "ingles", "informatic", "patent",
    "english", "computer", "license"

    # pattern temporali
    #"2019", "2020", "2021", "2022", "2023", "2024", "dal", "al",

    # aziende / enti
    "aziend", "ente", "enti", "srl", "spa", "cooperativ", "comun", "region"
    "regione", "municipality", "company", "organization", "corporation"
]

df_vacancy["IS_VALID_VACANCY"] = df_vacancy["DESCRIPTION"].str.lower().apply(
    lambda x: any(k in x for k in keywords) and len(x) > 50 if isinstance(x, str) else False
)

is_valid_vacancy = np.sum(df_vacancy["IS_VALID_VACANCY"])/df_vacancy.shape[0]

print(f"""Percentage of invalid vacancies (custom keywords matching): {1-is_valid_vacancy:.2%}""")




Percentage of invalid vacancies (custom keywords matching): 37.21%


In [27]:
general_info(df_vacancy)

Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35576 entries, 0 to 35575
Data columns (total 23 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   JOBORDER_ID_BH                       35576 non-null  int32         
 1   DATE_LAST_ACTIVE                     35576 non-null  datetime64[ns]
 2   JOBORDER_ID_PB                       16109 non-null  object        
 3   JOB_TITLE                            35576 non-null  object        
 4   POSITION_ESCO                        25618 non-null  object        
 5   RUOLO_LAVORATIVO_STANDARD_MPG        22380 non-null  object        
 6   CAP                                  29742 non-null  object        
 7   JOB_SITE                             4313 non-null   object        
 8   CCNL_LEVEL                           2848 non-null   object        
 9   LEGAL_CATEGORY                       2848 non-null   object        
 10  CCN

# COMMENTS on missing/unique values #

32024 vacancies

Comments on candidate missing data (Bullhorn):
-   jobtitle 0%
-   salary 0% 
-   citta 13%
-   regione 16%
-   skills 98%
-   parttime_perc 94%
-   testo_pubblicatione 30%
-   requisiti 47%
-   titoli_richiesti 99%
-   descrizione_uso_interno 30%
-   richieste_aggiuntive 99%
-   description (0.03% missing. this means that skills, testo_pubblicazione, requisiti, titoli_richiesti, descrizione_us_interno are simultaneously missing)

Comments on candidate missing data (PowerBase):
-   job_site 92%
-   job_position 92%
-   parttime_perc 97% missing
-   CCNL level 92% missing
-   CCNL code 92%
-   CCNL description 92%
-   legal_category 92%


Some comments on unique values:
- 12 unique values for titoli_richiesti
- 400 unique regions
- 162 unique CCNL codes
- 196 unique CCNL descriptions
- 103 unique CCNL levels
- 6 unique legal categories

In [29]:
"""
columns_to_check = ["TITOLI_RICHIESTI", "TESTO_PUBBLICAZIONE", "REQUISITI", "RICHIESTE_AGGIUNTIVE", "DESCRIZIONE_USO_INTERNO"]

unique_limit = 100
# Print unique values for each column
for col in columns_to_check:
    unique_values = df_vacancy[col].sort_values().unique()
    all_values = len(df_vacancy[col])
    nonmissing_values = len(df_vacancy[col].dropna())
    n_unique = len(unique_values)
    print(f"{n_unique} unique values in {col} over {nonmissing_values} total elements:")
    print("" if n_unique < unique_limit else f"(truncated to first {unique_limit})")

    print(unique_values[:min(unique_limit, n_unique)])
    print("-" * 40 + "\n")
"""


'\ncolumns_to_check = ["TITOLI_RICHIESTI", "TESTO_PUBBLICAZIONE", "REQUISITI", "RICHIESTE_AGGIUNTIVE", "DESCRIZIONE_USO_INTERNO"]\n\nunique_limit = 100\n# Print unique values for each column\nfor col in columns_to_check:\n    unique_values = df_vacancy[col].sort_values().unique()\n    all_values = len(df_vacancy[col])\n    nonmissing_values = len(df_vacancy[col].dropna())\n    n_unique = len(unique_values)\n    print(f"{n_unique} unique values in {col} over {nonmissing_values} total elements:")\n    print("" if n_unique < unique_limit else f"(truncated to first {unique_limit})")\n\n    print(unique_values[:min(unique_limit, n_unique)])\n    print("-" * 40 + "\n")\n'

In [30]:
columns_to_check = ["CCNL_LEVEL", "CCNL_CODE", "LEGAL_CATEGORY", "CCNL_DESCRIPTION"]

unique_limit = 110
# Print unique values for each column
for col in columns_to_check:
    unique_values = df_vacancy[col].sort_values().unique()
    all_values = len(df_vacancy[col])
    nonmissing_values = len(df_vacancy[col].dropna())
    n_unique = len(unique_values)
    print(f"{n_unique} unique values in {col} over {nonmissing_values} total elements:")
    print("" if n_unique < unique_limit else f"(truncated to first {unique_limit})")

    print(unique_values[:min(unique_limit, n_unique)])
    print("-" * 40 + "\n")



110 unique values in CCNL_LEVEL over 2848 total elements:
(truncated to first 110)
['1' '1 super q' '140' '2' '3' '3 ap liv. 1' '3 qufun inf' '3 super' '3/1'
 '3/2' '3/3' '3a' '4' '4 junior' '4 qufun inf' '4 senior' '4 super (ex 5)'
 '4/1' '4/3' '5' '5 (ex 6)' '5/0' '5/1' '5a' '6' '6 junior' '6 senior'
 '6 super' '6 super (ex 7)' '6i' '7' '8' '9' 'a' 'a1' 'a2' 'a3' 'ae1'
 'ae2' 'ae3' 'ap prima' 'ap seconda' 'ap terza' 'area 2' 'area 3'
 'area 3 liv. 1' 'area assistenti' 'area esecutiva' 'area istruttori'
 'area operatori' 'area supporto' 'b' 'b classe 1' 'b super' 'b/2' 'b/3'
 'b1' 'b1 super' 'b2' 'b2.' 'b3' 'b3.' 'b4' 'bs super' 'c' 'c super' 'c/1'
 'c/2' 'c/3' 'c0' 'c1' 'c2' 'c2 (ex d1)' 'c3' 'c4' 'd' 'd/1' 'd/2' 'd1'
 'd1 (ex 6)' 'd1 (ex d2)' 'd1s' 'd2' 'd2 (ex e)' 'd2s' 'd3' 'e' 'e1' 'e2'
 'e3' 'e4' 'eq e funzionari' 'ex 1-2 area' 'f' 'f 1 posiz.' 'f1' 'f2' 'g'
 'g1' 'h' 'i' 'junior' 'op base' 'op centrato' 'op consolidato'
 'op non prof.' 'operat esperti' 'prof.s. e funz.' 'quadro

In [36]:
b1 = bar_plot(df_vacancy, 'POSITION_ESCO', rotation=25)#, label_range=30)
b2 = bar_plot(df_vacancy, 'RUOLO_LAVORATIVO_STANDARD_MPG', rotation=25)#, label_range=30)
b3 = bar_plot(df_vacancy, 'RUOLO_LAVORATIVO_STANDARD_MPG_SPLIT', rotation=25)#, label_range=30)


display(b1)
display(b2)
display(b3)



:Bars   [POSITION_ESCO]   (COUNT)

:Bars   [RUOLO_LAVORATIVO_STANDARD_MPG]   (COUNT)

:Bars   [RUOLO_LAVORATIVO_STANDARD_MPG_SPLIT]   (COUNT)

In [32]:
b1 = bar_plot(df_vacancy, 'CITY', rotation=25)#, label_range=30)
b2 = bar_plot(df_vacancy, 'CAP', rotation=25)#, label_range=30)
b3 = bar_plot(df_vacancy, 'REGION', rotation=25)#, label_range=30)
    
(b1 + b2 + b3).cols(1)

:Layout
   .Bars.I   :Bars   [CITY]   (COUNT)
   .Bars.II  :Bars   [CAP]   (COUNT)
   .Bars.III :Bars   [REGION]   (COUNT)

In [33]:
b1 = bar_plot(df_vacancy, 'JOB_SITE', rotation=25)#, label_range=30)
display(b1)


:Bars   [JOB_SITE]   (COUNT)

In [ ]:
df1 = df_esco
df2 = df_vacancy
df1 = df_esco
df2 = df
df1 = df
df2 = df_vacancy

# Common values

c1 = 'POSITION ESCO'
c2 = 'POSITION_ESCO'
c1 = 'POSITION ESCO'
c2 = 'JOB_FIRST'
c1 = 'JOB_FIRST'
c2 = 'RUOLO_LAVORATIVO_STANDARD_MPG_SPLIT'

df1[c1] = df1[c1].str.lower()
df2[c2] = df2[c2].str.lower()

common = set(df1[c1].dropna()) & set(df2[c2].dropna())

# Percentage of values in each column that overlap with the other
overlap_c1 = df1[c1].isin(common).mean() * 100
overlap_c2 = df2[c2].isin(common).mean() * 100

print(f"Common unique values: {len(common)}")
print(f"% of candidate esco positions overlapping with vacancy esco positions: {overlap_c1:.2f}%")
print(f"% of vacancy esco positions overlapping with candidate esco positions: {overlap_c2:.2f}%")


col1 = df1[c1].dropna()
col2 = set(df2[c2].dropna())

# Count occurrences in df1 and whether each value exists in df2
result = (
    col1.value_counts()
    .rename_axis('value')
    .reset_index(name='count_in_df')
)

result['overlap'] = result['value'].isin(col2)

# Overlapping first, then non-overlapping, most frequent first
result = result.sort_values(
    ['overlap', 'count_in_df'],
    ascending=[False, False]
).reset_index(drop=True)

print(result[result['overlap'] == True])

NameError: name 'df_esco' is not defined

In [ ]:
df_esco = session.sql("SELECT * FROM it_discovery.landing.vacancytitle_esco").to_pandas()

In [ ]:
b1 = bar_plot(df, 'JOB_FIRST', rotation=25)#, label_range=30)
b2 = bar_plot(df_vacancy, 'POSITION_ESCO', rotation=25)#, label_range=30)
b3 = bar_plot(df_vacancy, 'RUOLO_LAVORATIVO_STANDARD_MPG', rotation=25)#, label_range=30)
b4 = bar_plot(df_vacancy, 'RUOLO_LAVORATIVO_STANDARD_MPG_SPLIT', rotation=25)#, label_range=30)

display(b1)
display(b2)
display(b3)
display(b4)



:Bars   [JOB_FIRST]   (COUNT)

:Bars   [POSITION_ESCO]   (COUNT)

:Bars   [RUOLO_LAVORATIVO_STANDARD_MPG]   (COUNT)

:Bars   [RUOLO_LAVORATIVO_STANDARD_MPG_SPLIT]   (COUNT)

In [ ]:
df_geo = session.sql("SELECT * FROM it_discovery.landing.geo_ita_mapping").to_pandas()

In [ ]:
df_geo = session.sql("SELECT * FROM it_discovery.ita_exp_onbench.mpg_it_onbench_italian_cities_test").to_pandas()

In [ ]:
general_info(df_geo)

Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14479 entries, 0 to 14478
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   UNIQUE_IDENTIFIER  14479 non-null  object 
 1   CITY_NAME          14479 non-null  object 
 2   PROVINCE           14479 non-null  object 
 3   PROVINCE_EXT       14479 non-null  object 
 4   REGION             14479 non-null  object 
 5   ZIP                14479 non-null  object 
 6   LATITUDE           14338 non-null  float64
 7   LONGITUDE          14336 non-null  float64
dtypes: float64(2), object(6)
memory usage: 905.1+ KB
None


Percentage of missing values:
UNIQUE_IDENTIFIER    0.00
CITY_NAME            0.00
PROVINCE             0.00
PROVINCE_EXT         0.00
REGION               0.00
ZIP                  0.00
LATITUDE             0.97
LONGITUDE            0.99
dtype: float64


Number of duplicated rows:
0


Check number of unique values for each categorical column 

In [ ]:
df_geo[df_geo["COMUNE"] == "Roma"]

,COMUNE,CAP,SIGLA_PROVINCIA,NOME_PROVINCIA,REGIONE,LATITUDINE,LONGITUDINE,TIMESTAMP
123,Roma,118,RM,Roma,Lazio,41.892724,12.4836647,2025-12-16 12:19:40.223104


In [ ]:
df_gj = session.sql(f"""select c.candidate_id_bh, c.city, c.cap, c.province, i.city_normalized, i.cap as icap, i.province_ext, i.region
                     from it_discovery.ita_exp_onbench.MPG_IT_ONBENCH_TEST_CANDIDATE_CLEANED_GEO c
                     left join it_discovery.ita_exp_onbench.MPG_IT_ONBENCH_TEST_ITALIAN_CITIES_JSON i
                     on c.city = i.city_normalized
                     --on c.cap = i.cap
                 """).to_pandas()



In [ ]:
df_gj = session.sql(f"""select c.candidate_id_bh, c.city, c.cap, c.province, c.city_lookup, c.cap_lookup, c.province_lookup, c.region_lookup
                     from it_discovery.ita_exp_onbench.MPG_IT_ONBENCH_TEST_CANDIDATE_CLEANED_GEO c
                 """).to_pandas()



In [ ]:
df_gj = session.sql(f"""select c.joborder_id_bh, c.city, c.cap, c.region, c.city_lookup, c.cap_lookup, c.province_lookup, c.region_lookup
                     from it_discovery.ita_exp_onbench.MPG_IT_ONBENCH_TEST_JOBORDER_CLEANED_GEO c
                 """).to_pandas()



In [ ]:
df_gj[df_gj["CITY_LOOKUP"].isna()]
df_gj[df_gj["CITY_LOOKUP"].isna() & ~df_gj["CITY"].isna()]


,JOBORDER_ID_BH,CITY,CAP,REGION,CITY_LOOKUP,CAP_LOOKUP,PROVINCE_LOOKUP,REGION_LOOKUP
22,970518,prov arezzo,52100,toscana,None,None,None,None
43,941899,roma lazio italia,00131,lazio,None,None,None,None
46,1050743,san pietro in casale bo,40018,emilia romagna,None,None,None,None
80,1052762,santa maria delle mole,00040,roma,None,None,None,None
121,1045139,bibione,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...
34584,1024487,ivrea to,10015,piemonte,None,None,None,None
34606,1016668,niviano,29029,emilia romagna,None,None,None,None
34608,1012824,san martino di castrozza,38054,veneto,None,None,None,None
34655,1047512,falciano,47891,None,None,None,None,None


In [ ]:
df_gj[df_gj["EXACT_MATCH"] == 0]


,CANDIDATE_ID_BH,CITY,CAP,PROVINCE,CITY_LOOKUP,CAP_LOOKUP,PROVINCE_LOOKUP,REGION_LOOKUP,EXACT_MATCH
6,3705550,None,None,None,Mergozzo,28802,vb,piemonte,0
12,4334337,None,None,None,Mergozzo,28802,vb,piemonte,0
27,4601952,None,None,None,Mergozzo,28802,vb,piemonte,0
31,3810165,vitinia,00127,rm,Vizzini,95049,ct,sicilia,0
42,2103167,marina di ginosa,74025,ta,Ari,66010,ch,abruzzo,0
54,3137146,alberoro,52048,ar,Alberona,71031,fg,puglia,0
80,3199283,None,None,None,Mergozzo,28802,vb,piemonte,0
151,4364572,reggio emilia,42124,re,Re,28856,vb,piemonte,0
177,5071242,None,None,None,Mergozzo,28802,vb,piemonte,0
182,4322537,introzzo valvarrone,23836,lc,Ne,16040,ge,liguria,0


In [ ]:
df_gj[df_gj["CITY_NORMALIZED"].isna() & ~ df_gj["CITY"].isna()]


KeyError: 'CITY_NORMALIZED'

In [ ]:
general_info(df_gj)

Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7896 entries, 0 to 7895
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CAP_LIST         7896 non-null   object 
 1   CAPOLUOGO        7896 non-null   bool   
 2   NOME             7896 non-null   object 
 3   NOME_NORMALIZED  7896 non-null   object 
 4   SIGLA            7896 non-null   object 
 5   CAP              7893 non-null   object 
 6   REGIONE          7896 non-null   object 
 7   PROVINCIA        7896 non-null   object 
 8   LAT              7893 non-null   float64
 9   LNG              7893 non-null   float64
dtypes: bool(1), float64(2), object(7)
memory usage: 563.0+ KB
None


Percentage of missing values:
CAP_LIST           0.00
CAPOLUOGO          0.00
NOME               0.00
NOME_NORMALIZED    0.00
SIGLA              0.00
CAP                0.04
REGIONE            0.00
PROVINCIA          0.00
LAT                0.04
LNG               